<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Why word order and context matter

**[Why word order and context matter](https://learning.nextia-ai.com/courses/deep-learning/m05/word-order-and-context/)** · Deep Learning and Transformers Explained · Module 5, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** prove that two tickets with the same words get the same bag of words, count word pairs and triples and measure what they cost, and list the validation tickets that the bag-of-words network routes wrongly, by style.

| | |
|---|---|
| **Time** | About 25 minutes, including 30 seconds of training |
| **Needs** | An internet connection for the first cells (about 2 MB of tickets). No account. Colab and Kaggle have PyTorch. |
| **You should know** | The bag of words, from [The forward pass](https://learning.nextia-ai.com/courses/deep-learning/m02/the-forward-pass/), and [Tokens and embeddings](https://learning.nextia-ai.com/courses/deep-learning/m05/tokens-and-embeddings/). |
| **You do not need** | Any earlier notebook of this course. |
| **Tested** | 2026-10-08, Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m05/word-order-and-context/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M05-L02-word-order-and-context/word-order-and-context-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Same words, same input

The two tickets below are **made up** for the lesson. They use the same 14 tokens in a different order. Grace's policy sends A to delivery (broken on arrival) and B to warranty (broken after use).

> **Predict.** Are their bags of words the same?

Run the cell. `torch.equal` is `True` when two tensors have the same shape and the same numbers.

In [ ]:
vocab = Vocab([r["text"] for r in train_rows])

a = "The lamp was damaged on arrival, not after a month of use."
b = "The lamp was damaged after a month of use, not on arrival."
xa = bag_of_words([{"text": a}], vocab)
xb = bag_of_words([{"text": b}], vocab)
print(xa.shape, "tokens:", int(xa.sum()), "| the same input:", torch.equal(xa, xb))

> **Check.** `the same input: True`. A model computes its output from its input only, so every model that reads these counts must give A and B the same answer.

> **Your turn.** Write two short tickets of your own with the same words in a different order, which Grace's policy sends to different teams. Store them in `mine_a` and `mine_b`. The check cell tests that their bags of words are identical and that no token is unknown.

In [ ]:
mine_a = ...
mine_b = ...

In [ ]:
if isinstance(mine_a, str) and isinstance(mine_b, str):
    same = torch.equal(bag_of_words([{"text": mine_a}], vocab), bag_of_words([{"text": mine_b}], vocab))
    unknown = [t for t in tokenize(mine_a + " " + mine_b) if t not in vocab.stoi]
    expect("the same bag of words", same, True)
    expect("unknown tokens", unknown, [])
else:
    print("Not yet: write your two tickets in the cell above, then run it again.")

## 2. Count pairs and triples (n-grams)

An n-gram is a run of n neighbouring tokens. The function below counts them.

> **Your turn.** For the lamp pair (`a`, `b`) and the password pair (`c`, `d`), find the smallest `n` for which the n-gram counts differ. Store them in `n_lamp` and `n_password`.

In [ ]:
from collections import Counter

def ngrams(text, n):
    t = tokenize(text)
    return Counter(tuple(t[i:i + n]) for i in range(len(t) - n + 1))

c = "The password problem is solved. The double charge on my card is still open."
d = "The double charge on my card is solved. The password problem is still open."
n_lamp = ...
n_password = ...

In [ ]:
expect_hash('n_lamp and n_password', [n_lamp, n_password], '5bf775530e2e5fcd')

In the password pair, "solved" and "still open" are two words away from "problem" and "card", so every word pair appears in both tickets. Whatever window you choose, a sentence can put the important word just outside it.

## 3. What pairs cost

> **Your turn.** Count every word and every word pair in the training tickets (use `ngrams(text, 1)` and `ngrams(text, 2)`), and store in `kept` the number of different ones that appear at least twice. Predict first: how many times larger than the 2,862 words is it?

In [ ]:
kept = ...

In [ ]:
expect('kept', kept, 19310)

With `<pad>` and `<unk>`, the vocabulary grows from 2,864 to 19,312 entries, and the bag-of-words network from 183,685 to 1,236,357 parameters, for the same 4,696 tickets.

## 4. Where the network fails

The next cell trains the bag-of-words network as in Module 3 (20 epochs, about 30 seconds) and counts its wrong validation tickets by `style`. The style is never an input; it only says how a ticket was written.

> **Predict.** Which style has the most errors: plain, request last, negation or two topics?

In [ ]:
from collections import Counter

x_train, x_valid = bag_of_words(train_rows, vocab), bag_of_words(valid_rows, vocab)
y_train, y_valid = labels(train_rows), labels(valid_rows)
set_seed(0)
net = BagOfWordsNet(len(vocab))
train(net, x_train, y_train, x_valid, y_valid, epochs=20, log=None)
net.eval()
with torch.no_grad():
    pred = net(x_valid).argmax(1)
wrong = [r for r, p in zip(valid_rows, pred.tolist()) if TEAMS[p] != r["team"]]
print(len(wrong), "wrong:", Counter(r["style"] for r in wrong).most_common())
for r in wrong:
    if r["style"] in ("negation", "two_topics"):
        print("-", r["ticket_id"], r["style"], "| Grace:", r["team"], "|", r["text"][:110], "...")

> **Check.** The lesson's run: 16 wrong tickets, 5 with two topics, 4 long, 3 on a boundary, 2 with a negation, 1 distractor, 1 short; none plain and none with the request last. Another computer can differ by a ticket or two.

Now ask the trained network about the made-up lamp tickets:

In [ ]:
with torch.no_grad():
    for name, x in (("A", xa), ("B", xb)):
        p = net(x).softmax(1)[0]
        print(name, {t: round(v, 3) for t, v in zip(TEAMS, p.tolist())})

> **Check.** A and B get exactly the same probabilities (the lesson's run: warranty 0.594, delivery 0.380). That is right for B and wrong for A, and no training can change it.

## Next

You have proved that a bag of words cannot see order, measured what word pairs cost, and found the tickets where the network fails. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m05/word-order-and-context/) has the diagram and the knowledge checks. The next lesson builds attention.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Attention step by step](https://learning.nextia-ai.com/courses/deep-learning/m05/attention-step-by-step/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m05/word-order-and-context/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets are made up for this course by a language model; no real customer wrote them.